# HyperParameter Tuning with Ray Tune and TensorBoard

Tuning neural network models is important, but it can be time consuming. We don't want to manually manage what needs to be trained, nor do we want to sit there and watch the training process. We can use some tools to manage this process and hopefully make it easier. 

## Tensorboard

Tensorboard is a visualization and monitoring tool that originally came from tensorflow, but can be used with PyTorch as well. Tensorboard can do several things, but we will focus on the basics:

1. Visualize training and validation metrics (loss, accuracy, etc.) over time.
2. Compare different training runs (e.g., different hyperparameter configurations).

Tensorboard runs a small web server that takes in information from a log directory and visualizes it on a few web pages. 

### Launching Tensorboard

One mild annoyance is that tensorboard has different ways to launch depending on the environment. In a Jupyter notebook, you can use the `%tensorboard` magic command, in vscode, you can use the %load_ext tensorboard magic command, and in a terminal, you can use the `tensorboard` command.

The launch command aims tensorboard at the log directory, it'll pull any logs from there. It will look through subdirectories by default, so you can use that to organize the logs. For example, we could do one trial with different versions of a CNN model, then other trials with different versions of an RNN model (later), and we can have each set organized in a different subdirectory.

<b>Note:</b> the differences in launching are things that all stem from using it in an interactive environment like our notebooks. In 'real life' it just runs as a service, and you access it via a web browser - all very standardized. 

In [ ]:
try:
    import torch_tb_profiler
except ImportError:
    !pip install torch_tb_profiler --quiet
    import torch_tb_profiler

log_root = "ray_results"
#create directory for tensorboard logs
import os as os2
if not os2.path.exists(log_root):
    os2.makedirs(log_root)
    
try:
    %tensorboard --logdir {log_root}
except:
    %reload_ext tensorboard
    %tensorboard --logdir {log_root}T

### Tensorboard Logging

Our code reports things to the tensorboard by writing logs in a specific format to a log directory. The tensorboard web server reads these logs and visualizes them. We can write logs to the tensorboard using the `SummaryWriter` class from the `torch.utils.tensorboard` module. 

The summary writer is basically a special command that writes our data to the log directory in a format that tensorboard can read. There are several different types of logs we can write, including:
<ul>
<li>Scalar: A single value that changes over time (e.g., loss, accuracy, etc.)</li>
<li>Histogram: A distribution of values (e.g., weights, activations, etc.)</li>
<li>Image: A single image or a batch of images (e.g., input images, feature maps, etc.)</li>
<li>Text: A string of text (e.g., model architecture, hyperparameters, etc.)</li>
</ul>

Note that if you're showing examples, such as images or text, the tensorboard won't do something like automatically add all the images and allow you to scroll through them. If getting examples, it is usally easiest to supply a set of examples that will be visualized each time, and write some code to process and log the version you want to display. This can be something like predicting the class of a sample image, and then showing the image with the predicted class as the title, generating a sample piece of text in a generative model, or producing images like the saliency or kernel visualizations in the CNN models. 

#### Redundant Logging (in Here)

The ray.reporter will log the results of each trial, and by default it will also log the results in a format that tensorboard captures. In this example, I have redundant logging to tensorboard, so we can see how it is done the "manual" way. We don't need this, in a real example I would leave the ray.reporter logging, and remove the manual tensorboard logging.

The logging action can write almost anything you can imagine to the logs, which will then be showed in tensorboard. 

## Ray Tune

Ray Tune is a library for hyperparameter tuning that allows us to easily run multiple training runs with different hyperparameter configurations, and monitor the results. It provides several features, including:
- Tuning with hyperparameter search spaces (e.g., grid search, random search, Bayesian optimization, etc.)
- Early stopping of trials that are not performing well (e.g., using ASHA scheduler)
- Distributed training across multiple machines and GPUs

## Really Large Loops

When using RayTune, we need to create several variations of our models, and train each one to monitor the performance. The loop here is very large as it wraps around the entire training process, and it is responsible for building the model, training it, and reporting the results to RayTune.

<b>Our ultimate goal here is to create a function that takes in an argument, the configuration, which is a dictionary that contains the hyperparameters for that trial. The function will then build the model based on the configuration, train it, and report the results to RayTune. This function will be called by RayTune for each trial, and it will be responsible for managing the entire training process for that trial.</b>

### Search Space and Model Building

The process will work mostly like a grid search, but a slightly more involved one. Each of the "config" parameters in the model building function will be set to a value from the config dictionary below when the model for that iteration is built.

The search space is defined in the config dictionary, and it specifies the hyperparameters that we want to tune along with the options. This is basically like a grid search, but with more flexibility. The search space lists hyperparameters and some definition of how to select values for those hyperparameters. The tuning process will then try different combinations of those hyperparameters and monitor the results.

There are several types of choices we can set up in the config dictionary, including:
- `tune.choice`: This allows us to specify a list of discrete values to choose from. For example, we can specify a list of possible numbers of layers or channels.
- `tune.uniform`: This allows us to specify a range of continuous values to choose from. For example, we can specify a range of dropout rates or learning rates.
- `tune.loguniform`: This is similar to `tune.uniform`, but it samples values on a logarithmic scale. This is often used for learning rates, which can vary over several orders of magnitude.
- `tune.randint`: This allows us to specify a range of integer values to choose from. For example, we can specify a range of batch sizes.
- `tune.sample_from`: This allows us to specify a custom sampling function that generates values to be used in the tuning process. This can be useful for more complex sampling strategies that aren't covered by the other options.

Defining the search space in the config dictionary is more or less the same as setting up a grid search, but it allows us to vary almost anything we can imagine. 

### Scheduler

The scheduler is responsible for deciding which trials to continue and which to stop based on the reported metrics. In this example, we are using the ASHA scheduler, which is an early stopping algorithm that stops trials that are not performing well. The scheduler monitors the "accuracy" metric and tries to maximize it.

There are other schedulers available, but we won't really focus on them. ASHA is recommended as a good default choice for most cases.

#### Tune Reports 

At the end of each epoch, we report the validation loss and accuracy to RayTune using `tune.report()`. This allows RayTune to keep track of the performance of each trial and make decisions about which trials to continue or stop based on the reported metrics. The metrics we deliver here are what the process monitors, we select that in the scheduler below.

### RayTune Concerns

There are a few things to keep in mind when using RayTune:
- The training function needs to be serializable, which means it cannot have any non-serializable objects - largely things that are "reaching out" from the code to the outside world. This includes things like file handles, database connections, etc... 
- In notebooks, there are some issues with raytune accessing the resources (e.g. disk or gpu) that the notebook is using. I found some notes on this, it isn't a massive issue, but the internet recommended putting things in one cell, so I defaulted to that. Some runs failed to grab the GPU, for no clear reason, it appears to just be a bug - in 'real' scenarios, this wouldn't be done in notebooks, so I doubt that it is the most critical concern. 

#### Serializable

One potential challenge that raytune encounters is that the training function needs to be serializable, which means it cannot have any non-serializable objects - largely things that are "reaching out" from the code to the outside world. This includes things like file handles, database connections, etc...

For our purposes, this isn't super important, but it becomes more important if we are trying to do something larger. Many larger models aren't trained on one GPU, they are trained on, potentially, thousands of GPUs installed on thousands of machines. If we are distributing the training across many machines, that serialization is critical, it allows the training function to be sent to the different machines as 'whole' objects that can be executed without needing to worry about anything external. 

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import torchvision
try:
    from ray import tune
except ImportError:
    !pip install ray[tune] --quiet
    from ray import tune
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler
from ray.air import session
from torch.utils.tensorboard import SummaryWriter
from ray.tune import Tuner, TuneConfig
from ray.air import RunConfig

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

# ---- Data Loading Function ----
def load_data(data_dir="./data"):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    trainset = torchvision.datasets.CIFAR10(
        root=data_dir, train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(
        root=data_dir, train=False, download=True, transform=transform_test)
    return trainset, testset


# ---- CNN Model ----
class CifarCNN(nn.Module):
    def __init__(self, num_filters_1, num_filters_2, fc_size, dropout_rate):
        super(CifarCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, num_filters_1, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_filters_1)
        self.conv2 = nn.Conv2d(num_filters_1, num_filters_2, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_filters_2)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(num_filters_2 * 8 * 8, fc_size)
        self.fc2 = nn.Linear(fc_size, 10)
    # Example of a convoluted forward pass, we could do it step by step. 
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# ---- Custom Training Function ----
def train_cifar(config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = CifarCNN(
        num_filters_1=config["num_filters_1"],
        num_filters_2=config["num_filters_2"],
        fc_size=config["fc_size"],
        dropout_rate=config["dropout_rate"],
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"],
                           weight_decay=config["weight_decay"])

    trainset, testset = load_data()

    trainloader = DataLoader(trainset, batch_size=config["batch_size"],
                             shuffle=True, num_workers=2)
    testloader = DataLoader(testset, batch_size=256,
                            shuffle=False, num_workers=2)

    # TensorBoard writer for this trial
    writer = SummaryWriter(log_dir=os.path.join(session.get_trial_dir(), "tb_logs"))

        # --- BEGIN PROFILING SETUP ---
    profiler_log_dir = os.path.join(session.get_trial_dir(), "profiler_logs")
    os.makedirs(profiler_log_dir, exist_ok=True)

    profiler = torch.profiler.profile(
        activities=[
            torch.profiler.ProfilerActivity.CPU,
        ] + ([torch.profiler.ProfilerActivity.CUDA] if torch.cuda.is_available() else []),
        schedule=torch.profiler.schedule(
            wait=1,   # skip the first batch (warmup)
            warmup=1, # warmup for 1 batch
            active=3, # profile 3 batches
            repeat=1, # only profile once per epoch cycle
        ),
        on_trace_ready=torch.profiler.tensorboard_trace_handler(profiler_log_dir),
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
    )
    profiler.start()
    # --- END PROFILING SETUP ---

    for epoch in range(20):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        with torch.enable_grad():
            for inputs, labels in trainloader:
                profiler.step()  # Step the profiler at the start of each batch
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total

        # Log to TensorBoard
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Accuracy/train", train_acc, epoch)
        writer.add_scalar("Accuracy/val", val_acc, epoch)
        writer.add_histogram("Model/conv1_weights", model.conv1.weight, epoch)
        writer.add_histogram("Model/conv2_weights", model.conv2.weight, epoch)
        writer.add_histogram("Model/fc1_weights", model.fc1.weight, epoch)
        writer.add_histogram("Model/fc2_weights", model.fc2.weight, epoch)
        writer.flush()

        # Report to Ray Tune
        session.report({"loss": val_loss, "accuracy": val_acc,
                        "train_loss": train_loss, "train_accuracy": train_acc})
    # --- BEGIN PROFILER CLEANUP ---
    profiler.stop()
    # --- END PROFILER CLEANUP ---
    writer.add_hparams(config, {"final_accuracy": val_acc})
    writer.close()

# RAYTUNE TRIAL CONFIGURATION
# ---- Custom Tunable Search Space ----
def custom_lr(spec):
    """Sample learning rate on a log scale."""
    return 10 ** np.random.uniform(-5, -2)
def custom_weight_decay(spec):
    """Sample weight decay on a log scale."""
    return 10 ** np.random.uniform(-6, -3)
def custom_dropout(spec):
    """Sample dropout rate uniformly."""
    return np.random.uniform(0.0, 0.5)

# ---- Search Space Config ----
search_space = {
    "num_filters_1": tune.choice([16, 32, 64]),
    "num_filters_2": tune.choice([32, 64, 128]),
    "fc_size": tune.choice([128, 256, 512]),
    "dropout_rate": tune.sample_from(custom_dropout),
    "lr": tune.sample_from(custom_lr),
    "weight_decay": tune.sample_from(custom_weight_decay),
    "batch_size": tune.choice([32, 64, 128]),}

# ---- Scheduler ----
scheduler = ASHAScheduler(
    metric="accuracy",
    mode="max",
    max_t=8, # maximum number of epochs to train - will override loop
    grace_period=2, # minimum number of epochs to run before stopping
    reduction_factor=2,) #this controls how aggressively we stop trials, higher means more aggressive

# ---- Reporter ----
#this manages the output to the terminal, it will show the metrics we select
# the max_report_frequency controls how often the reporter updates.
reporter = CLIReporter(
    metric_columns=["loss", "accuracy", "train_loss", "train_accuracy",
                     "training_iteration"],
    max_report_frequency=30,)
# ---- Download data before trials start ----
load_data()

# ---- Run Ray Tune using Tuner API ----
num_gpus = 1 if torch.cuda.is_available() else 0
print(f"Using {'GPU' if num_gpus else 'CPU'} for training")

# ---- Launch TensorBoard ----
log_dir = os.path.abspath("./ray_results")
os.makedirs(log_dir, exist_ok=True)
os.environ.pop("AIR_VERBOSITY", None)

tuner = Tuner(
    tune.with_resources(train_cifar, {"cpu": 2, "gpu": num_gpus}),
    param_space=search_space,
    tune_config=TuneConfig(
        #metric="accuracy",
        #mode="max",
        num_samples=8,
        scheduler=scheduler,
    ),
    run_config=RunConfig(
        name="cifar10_cnn_hpsearch",
        storage_path=log_dir,
        progress_reporter=reporter,
    ),
)
result = tuner.fit()


In [ ]:
# ---- Best Trial Results ----
best_trial = result.get_best_result("accuracy", "max")
metrics_df = result.get_dataframe()
metrics_df.sort_values("accuracy", ascending=False).head()

## Exercise - Try One

Use the dataset above, or any other, to try a model with raytune hyperparameter tuning. 

In [ ]:
#